In [1]:
import time
import numpy as np
from Tokenizer import Tokenizer

def benchmark_tokenizer(tokenizer, text_samples, num_runs=3):
    """Benchmark tokenizer throughput"""
    
    # Combine all text samples
    full_text = '\n'.join(text_samples)
    text_bytes = len(full_text.encode('utf-8'))
    
    print(f"Benchmarking tokenizer with {text_bytes:,} bytes of text...")
    print(f"Text length: {len(full_text):,} characters")
    
    times = []
    for run in range(num_runs):
        start_time = time.time()
        tokens = tokenizer.encode(full_text)
        end_time = time.time()
        
        elapsed = end_time - start_time
        times.append(elapsed)
        
        print(f"Run {run+1}: {elapsed:.3f}s, {len(tokens):,} tokens")
    
    # Calculate statistics
    avg_time = np.mean(times)
    std_time = np.std(times)
    
    # Calculate throughput
    bytes_per_second = text_bytes / avg_time
    chars_per_second = len(full_text) / avg_time
    
    return {
        'avg_time': avg_time,
        'std_time': std_time,
        'bytes_per_second': bytes_per_second,
        'chars_per_second': chars_per_second,
        'text_bytes': text_bytes,
        'num_tokens': len(tokens) if 'tokens' in locals() else 0
    }

owt_tokenizer = Tokenizer.from_files(
    "../tokenizer_vocab_owt_32k.json", 
    "../tokenizer_merges_owt_32k.txt",
    special_tokens=["<|endoftext|>"]
)

print("Tokenizer loaded successfully!")


Tokenizer loaded successfully!


In [2]:
from datasets import load_dataset

# Load the OpenWebText dataset in streaming mode
print("Loading OpenWebText dataset...")
ds = load_dataset("Skylion007/openwebtext", streaming=True, split="train")

# Take a small sample for testing (first 100 examples)
sample_texts = []
print("Collecting sample texts...")

for i, example in enumerate(ds):
    if i >= 100:  # Only take first 100 examples
        break
    sample_texts.append(example['text'])
    if i % 20 == 0:
        print(f"Collected {i+1} texts...")

print(f"Collected {len(sample_texts)} sample texts")

# Show some stats about the collected texts
total_chars = sum(len(text) for text in sample_texts)
print(f"Total characters in sample: {total_chars:,}")
print(f"Average text length: {total_chars / len(sample_texts):.1f} characters")


Loading OpenWebText dataset...
Collected 1 texts...
Collected 21 texts...
Collected 41 texts...
Collected 61 texts...
Collected 81 texts...
Collected 100 sample texts
Total characters in sample: 500,830
Average text length: 5008.3 characters


In [5]:
# Benchmark the tokenizer on the OpenWebText sample
print("\n" + "="*50)
print("BENCHMARKING TOKENIZER ON OPENWEBTEXT SAMPLE")
print("="*50)

results = benchmark_tokenizer(owt_tokenizer, sample_texts, num_runs=3)

print(f"\nBenchmark Results:")
print(f"Average time: {results['avg_time']:.3f} ± {results['std_time']:.3f} seconds")
print(f"Throughput: {results['bytes_per_second'] / 1024 / 1024:.2f} MB/s")
print(f"Character throughput: {results['chars_per_second']:,.0f} chars/s")
print(f"Total tokens generated: {results['num_tokens']:,}")
print(f"Tokens per second: {results['num_tokens'] / results['avg_time']:,.0f}")



BENCHMARKING TOKENIZER ON OPENWEBTEXT SAMPLE
Benchmarking tokenizer with 505,877 bytes of text...
Text length: 500,929 characters
Run 1: 1.840s, 113,377 tokens
Run 2: 1.116s, 113,377 tokens
Run 3: 0.994s, 113,377 tokens

Benchmark Results:
Average time: 1.317 ± 0.373 seconds
Throughput: 0.37 MB/s
Character throughput: 380,378 chars/s
Total tokens generated: 113,377
Tokens per second: 86,092


In [13]:
print(f"It takes {825*1024/0.37/3600/24:.2f} days to encode the openwebtext dataset")

It takes 26.43 days to encode the openwebtext dataset
